<a href="https://colab.research.google.com/github/alkadevirajbanshi-ai/Intelligence-System/blob/main/Customer_Support_RAG_Completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7: Tutorial- Customer Support RAG Agent

**Learning outcomes:** upload documents, extract text, chunk, embed, store vectors in **ChromaDB**, retrieve supporting passages, and generate a grounded answer with a **free local model**.

**Runtime:** Google Colab → Runtime → Change runtime type → CPU (works, slower) or T4 GPU (faster, if available). Run cells from top to bottom. Model downloads require internet during setup. All inference is local within the Colab runtime.

**Important:** This is a *read-only customer-support RAG assistant*. It does not execute refunds, book services, access real accounts or create tickets. Uploaded documents and example records should not contain private customer information.

## Part A — Install packages
We use `pypdf` for PDF text extraction, `sentence-transformers` for embeddings, `chromadb` for vector search and `transformers` for local answer generation.

In [2]:
!pip -q install "chromadb>=0.5,<2" "sentence-transformers>=3,<6" "transformers>=4.45,<5" "reportlab>=4,<5" "pypdf>=5,<7" "sentencepiece>=0.2"

## Part B — Imports and model configuration
`all-MiniLM-L6-v2` turns text into vectors. `google/flan-t5-base` generates a response using retrieved evidence. Both are free, locally run Hugging Face models (downloaded on first use).

In [3]:
import os, re, uuid, json, torch
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import chromadb

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = "google/flan-t5-base"
CHUNK_WORDS = 110
CHUNK_OVERLAP = 25
TOP_K = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running locally on:", DEVICE)


Running locally on: cuda


## Part C — Prepare customer-support documents


Choose

**S** (built-in sample documents),

**U** (upload one or more TXT/MD/PDF files),

**B** (combine uploaded documents with the sample data).

The built-in sample documents are fictional. PDF support is limited to PDFs containing selectable text;

In [4]:
SAMPLE_DOCS = {
    "returns_policy.txt": """Customer Support Returns Policy. Customers may request a return within 30 days of delivery. Products must be in their original condition. To start a return, contact support with the order number and reason. Refunds are issued after inspection and may take 5 to 10 business days. Items damaged on arrival should be reported within 7 days of delivery. The agent cannot approve refunds automatically.""",
    "shipping_policy.txt": """Shipping and Delivery. Standard shipping normally takes 3 to 5 business days after dispatch. Express shipping normally takes 1 to 2 business days after dispatch. Shipping times are estimates, not guarantees. To check an individual order, customers must use the official order tracking portal. This demonstration has no access to live order tracking.""",
    "warranty_policy.txt": """Warranty and Repairs. Electronic devices have a limited 12-month warranty against manufacturing defects. The warranty excludes accidental damage and normal wear. Customers should retain proof of purchase. Contact support for warranty assessment. The agent cannot determine eligibility or authorize a replacement.""",
    "headphones_guide.txt": """Headphones Troubleshooting. If Bluetooth pairing fails, charge the headphones, turn Bluetooth off and on, and restart the phone. Hold the pairing button for 5 seconds until the LED flashes. Remove previous pairings and try again. If the device still cannot pair, contact support with the product model.""",
}

SUPPORTED_EXTENSIONS = {".txt", ".md", ".pdf"}

def upload_or_sample_documents(choice=None):
    """Return list of (filename, file_bytes) for sample, uploaded, or both."""
    if choice is None:
        choice = input("Choose S = sample (default), U = upload, B = both: ").strip().lower() or "s"
    choice = choice.lower()
    if choice not in {"s", "u", "b"}:
        raise ValueError("Choose 's', 'u', or 'b'.")

    selected = []
    if choice in {"s", "b"}:
        selected.extend((name, text.encode("utf-8")) for name, text in SAMPLE_DOCS.items())

    if choice in {"u", "b"}:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Uploading from a browser requires Google Colab; use 's' outside Colab.") from exc
        uploaded = files.upload()
        selected.extend((name, data) for name, data in uploaded.items())

    selected = [(name, data) for name, data in selected
                if Path(name).suffix.lower() in SUPPORTED_EXTENSIONS]
    if not selected:
        raise ValueError("No supported files were selected. Upload PDF, TXT or MD files, or choose S.")
    return selected

raw_files = upload_or_sample_documents()
print("Selected sources:", [name for name, _ in raw_files])

Choose S = sample (default), U = upload, B = both: b


Saving Grocery.csv to Grocery.csv
Selected sources: ['returns_policy.txt', 'shipping_policy.txt', 'warranty_policy.txt', 'headphones_guide.txt']


## Part D — Extract text from uploaded files
PDF extraction works on text-based PDFs; plain text and Markdown files are decoded as UTF-8.

In [6]:
from io import BytesIO
from pathlib import Path
import re
from pypdf import PdfReader

def extract_text(filename: str, content: bytes) -> str:
    """Convert TXT, MD, or selectable-text PDF into plain text."""
    extension = Path(filename).suffix.lower()
    if extension in {".txt", ".md"}:
        return content.decode("utf-8-sig", errors="replace")
    if extension == ".pdf":
        try:
            reader = PdfReader(BytesIO(content))
            pages = [(page.extract_text() or "") for page in reader.pages]
            return "\n".join(pages)
        except Exception as exc:
            print(f"Could not read PDF {filename}: {exc}")
            return ""
    print(f"Unsupported file type: {filename}")
    return ""

def prepare_documents(raw_files):
    """Create a uniform document list for the RAG pipeline."""
    prepared = []
    for filename, content in raw_files:
        extracted = extract_text(filename, content)
        cleaned = re.sub(r"\s+", " ", extracted).strip()
        if cleaned:
            prepared.append({"source": filename, "text": cleaned})
            print(f"Loaded {filename}: {len(cleaned.split())} words")
        else:
            print(f"Skipped {filename}: no selectable text extracted")
    if not prepared:
        raise ValueError("No readable documents. Use a text-based PDF, TXT, MD, or built-in samples.")
    return prepared

documents = prepare_documents(raw_files)
print("Total documents ready for chunking:", len(documents))

Loaded returns_policy.txt: 64 words
Loaded shipping_policy.txt: 53 words
Loaded warranty_policy.txt: 41 words
Loaded headphones_guide.txt: 47 words
Total documents ready for chunking: 4


### Document verification — test all supported input types

This optional check uses a generated **text-based PDF in memory**, plus a TXT and an MD file, to verify the extraction stage without uploading files. It also verifies that the built-in sample data can be prepared.

In [7]:
# Optional: run to verify the loader without changing `documents`.
from io import BytesIO
from reportlab.pdfgen import canvas

pdf_buffer = BytesIO()
pdf_canvas = canvas.Canvas(pdf_buffer)
pdf_canvas.drawString(72, 750, "PDF test: Refunds take 5 to 10 business days.")
pdf_canvas.showPage()
pdf_canvas.save()

check_files = [
    ("test.txt", b"TXT test: Standard shipping takes 3 to 5 business days."),
    ("test.md", b"# Markdown test\nWarranty is 12 months."),
    ("test.pdf", pdf_buffer.getvalue()),
]
check_documents = prepare_documents(check_files)
assert len(check_documents) == 3
assert "Refunds" in check_documents[2]["text"]
assert "Warranty" in check_documents[1]["text"]
assert len(prepare_documents([(k, v.encode("utf-8")) for k, v in SAMPLE_DOCS.items()])) == 4
print("PASS: TXT, MD, PDF, and original built-in samples can all be processed.")

Loaded test.txt: 10 words
Loaded test.md: 7 words
Loaded test.pdf: 9 words
Loaded returns_policy.txt: 64 words
Loaded shipping_policy.txt: 53 words
Loaded warranty_policy.txt: 41 words
Loaded headphones_guide.txt: 47 words
PASS: TXT, MD, PDF, and original built-in samples can all be processed.


## Part E — Chunk the text
A **chunk** is a short passage. Overlap preserves context at boundaries. Change `CHUNK_WORDS` and `CHUNK_OVERLAP` to observe how retrieval changes.

In [8]:
def split_into_chunks(text, chunk_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    if not 0 <= overlap < chunk_words:
        raise ValueError("Overlap must be smaller than chunk size")
    words = text.split()
    step = chunk_words - overlap
    return [" ".join(words[i:i + chunk_words]) for i in range(0, len(words), step) if words[i:i + chunk_words]]

chunks = []
for doc in documents:
    for idx, chunk in enumerate(split_into_chunks(doc["text"])):
        chunks.append({"id": str(uuid.uuid4()), "source": doc["source"], "part": idx + 1, "text": chunk})
print("Documents:", len(documents), "| Chunks:", len(chunks))
print("\nExample chunk:\n", chunks[0]["text"][:450])

Documents: 4 | Chunks: 4

Example chunk:
 Customer Support Returns Policy. Customers may request a return within 30 days of delivery. Products must be in their original condition. To start a return, contact support with the order number and reason. Refunds are issued after inspection and may take 5 to 10 business days. Items damaged on arrival should be reported within 7 days of delivery. The agent cannot approve refunds automatically.


## Part F — Create embeddings and store in ChromaDB
The embedding model produces numerical vectors. ChromaDB stores those vectors and the source text for similarity search. This notebook uses a fresh **in-memory** Chroma collection for predictable classroom reruns.

In [9]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
texts = [c["text"] for c in chunks]
vectors = embedding_model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()
print("Number of vectors:", len(vectors), "| Vector dimensions:", len(vectors[0]))

chroma_client = chromadb.EphemeralClient()
collection = chroma_client.get_or_create_collection(
    name="customer_support_demo",
    metadata={"hnsw:space": "cosine"},
)
# Clear existing items if this cell is rerun in the same runtime.
existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])
collection.add(
    ids=[c["id"] for c in chunks],
    documents=texts,
    metadatas=[{"source": c["source"], "part": c["part"]} for c in chunks],
    embeddings=vectors,
)
print("Chunks indexed in ChromaDB:", collection.count())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of vectors: 4 | Vector dimensions: 384
Chunks indexed in ChromaDB: 4


## Part G — Retrieve relevant passages
A user question is embedded using the **same embedding model**. Chroma returns the closest chunks. With cosine distance, **smaller distance means more similar**.

In [10]:
def retrieve(question, top_k=TOP_K):
    query_vector = embedding_model.encode([question], normalize_embeddings=True).tolist()[0]
    results = collection.query(query_embeddings=[query_vector], n_results=min(top_k, collection.count()), include=["documents", "metadatas", "distances"])
    return [
        {"text": txt, "source": meta["source"], "part": meta["part"], "distance": round(float(dist), 3)}
        for txt, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]

question = "How long does a refund take?"
for result in retrieve(question):
    print(f"{result['source']} | cosine distance={result['distance']} | {result['text'][:180]}...")

returns_policy.txt | cosine distance=0.323 | Customer Support Returns Policy. Customers may request a return within 30 days of delivery. Products must be in their original condition. To start a return, contact support with th...
shipping_policy.txt | cosine distance=0.648 | Shipping and Delivery. Standard shipping normally takes 3 to 5 business days after dispatch. Express shipping normally takes 1 to 2 business days after dispatch. Shipping times are...
warranty_policy.txt | cosine distance=0.849 | Warranty and Repairs. Electronic devices have a limited 12-month warranty against manufacturing defects. The warranty excludes accidental damage and normal wear. Customers should r...


## Part H — Load the free local language model
`flan-t5-base` is a small instruction-following model that runs without an API key. On CPU it can be slower; a T4 GPU speeds it up. Because its context window is limited, retrieved passages are truncated before generation.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
local_llm = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL).to(DEVICE)
local_llm.eval()
print("Local generator ready:", GENERATION_MODEL)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Local generator ready: google/flan-t5-base


## Part I — Complete RAG customer-support pipeline

1. Retrieve the top matching chunks.
2. Check whether there is enough relevant evidence.
3. Assemble a grounded prompt.
4. Generate the response locally.
5. Display the sources used.

**Safety rule:** If the retrieved information does not support an answer, the assistant should say it does not know rather than invent a policy. Similarity is an imperfect signal, so the threshold is a teaching heuristic, not a guarantee.

In [12]:
MAX_CONTEXT_TOKENS = 340
DISTANCE_CUTOFF = 0.72  # Teaching heuristic; adjust after inspecting retrieval scores.

def answer_customer(question, top_k=TOP_K, show_retrieval=True):
    hits = retrieve(question, top_k=top_k)
    if not hits or hits[0]["distance"] > DISTANCE_CUTOFF:
        return {"answer": "I couldn't find enough information in the uploaded documents. Please contact customer support.", "sources": [], "retrieved": hits}

    context = "\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    # Reserve input tokens for the question and instruction.
    context_ids = tokenizer.encode(context, add_special_tokens=False)[:MAX_CONTEXT_TOKENS]
    context = tokenizer.decode(context_ids, skip_special_tokens=True)
    prompt = (
        "You are a customer support assistant. Answer the question using ONLY the context. "
        "If the answer is missing, say: I do not know from the provided documents. "
        "Do not claim to have taken actions. Keep the answer short.\n\n"
        f"Context: {context}\n\nQuestion: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        output_ids = local_llm.generate(**inputs, max_new_tokens=100, num_beams=2, do_sample=False)
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    if not answer:
        answer = "I do not know from the provided documents."
    sources = list(dict.fromkeys(h["source"] for h in hits))
    if show_retrieval:
        print("Retrieved evidence:")
        for h in hits:
            print(f"  {h['source']} (distance={h['distance']})")
    return {"answer": answer, "sources": sources, "retrieved": hits}

result = answer_customer("How long does a refund take?")
print("\nAgent:", result["answer"])
print("Sources:", ", ".join(result["sources"]) or "none")

Retrieved evidence:
  returns_policy.txt (distance=0.323)
  shipping_policy.txt (distance=0.648)
  warranty_policy.txt (distance=0.849)

Agent: 5 to 10 business days
Sources: returns_policy.txt, shipping_policy.txt, warranty_policy.txt


## Part J — Try different customer questions
These examples are grounded in the fictional sample documents. If you uploaded your own files, change the questions to match their contents.

In [13]:
TEST_QUESTIONS = [
    "How do I return a product?",
    "How long does standard shipping take?",
    "What is the warranty period?",
    "How can I fix Bluetooth pairing?",
    "Can you refund order 123 right now?",
]
for q in TEST_QUESTIONS:
    result = answer_customer(q, show_retrieval=False)
    print("\nCustomer:", q)
    print("Agent:", result["answer"])
    print("Retrieved sources:", ", ".join(result["sources"]) or "none")


Customer: How do I return a product?
Agent: contact support with the order number and reason
Retrieved sources: returns_policy.txt, warranty_policy.txt, headphones_guide.txt

Customer: How long does standard shipping take?
Agent: 3 to 5 business days
Retrieved sources: shipping_policy.txt, returns_policy.txt, warranty_policy.txt

Customer: What is the warranty period?
Agent: 12-month
Retrieved sources: warranty_policy.txt, returns_policy.txt, headphones_guide.txt

Customer: How can I fix Bluetooth pairing?
Agent: charge the headphones, turn Bluetooth off and on, and restart the phone. Hold the pairing button for 5 seconds until the LED flashes. Remove previous pairings and try again. If the device still cannot pair, contact support with the product model. [returns_policy.txt] Customer Support Returns Policy. Customers may request a return within 30 days of delivery. Products must be in their original condition. To start a return, contact support with the order
Retrieved sources: headp

## Part K — Interactive chat (optional)
Run this cell, type a question, and enter `quit` to stop. This chat is **read-only** and has no account-specific memory or tool execution.

In [14]:
def customer_support_chat():

    print("=" * 60)
    print("        Customer Support RAG Chatbot")
    print("=" * 60)
    print("Ask questions about your uploaded or sample documents.")
    print("Type 'quit' or 'exit' to end the conversation.\n")

    while True:

        # Step 1: Get user question
        question = input("You: ").strip()

        # Step 2: Exit condition
        if question.lower() in ["quit", "exit"]:
            print("\nChat ended. Thank you!")
            break

        if not question:
            print("Please enter a question.\n")
            continue

        # Step 3: Send question to existing RAG agent
        try:
            result = answer_customer(
                question,
                top_k=TOP_K,
                show_retrieval=False
            )

            # Step 4: Display generated answer
            print("\nAgent:", result["answer"])

            # Step 5: Display retrieved document sources
            sources = result.get("sources", [])

            if sources:
                print("Sources:", ", ".join(sources))
            else:
                print("Sources: No relevant documents found")

            print("-" * 60)

        except Exception as e:
            print("Error:", str(e))


# Start chatting with the agent
customer_support_chat()

        Customer Support RAG Chatbot
Ask questions about your uploaded or sample documents.
Type 'quit' or 'exit' to end the conversation.

You: How do I return the product?

Agent: contact support with the order number and reason
Sources: returns_policy.txt, headphones_guide.txt, warranty_policy.txt
------------------------------------------------------------
You: How long does standard shipping take?

Agent: 3 to 5 business days
Sources: shipping_policy.txt, returns_policy.txt, warranty_policy.txt
------------------------------------------------------------
You: What is the warranty period?

Agent: 12-month
Sources: warranty_policy.txt, returns_policy.txt, headphones_guide.txt
------------------------------------------------------------
You:  How can I fix Bluetooth pairing?

Agent: charge the headphones, turn Bluetooth off and on, and restart the phone. Hold the pairing button for 5 seconds until the LED flashes. Remove previous pairings and try again. If the device still cannot pai

# Part L — Student Tasks (5 marks)

**Exercise L1 — Upload your own knowledge base (1 marks).** Upload two text or text-based PDF documents representing customer support FAQs or product manuals. Show extracted document names and chunk counts.

**Exercise L2 — Explore chunking (1 marks).** Change `CHUNK_WORDS` from 110 to 60, rerun chunking, embedding and indexing, and compare the number of chunks and top retrieved source for the same question. Explain one advantage and one disadvantage of smaller chunks.

**Exercise L3 — Explore retrieval (1 marks).** Compare `TOP_K=1` and `TOP_K=3` on two questions. Record the retrieved passages and explain whether more passages improved the answer.

**observation:** More context can help but may add irrelevant content or exceed the generator's context limit.

**Exercise L4 — Test unsupported requests (1 marks).** Ask the assistant for a policy that does not appear in the documents, or ask it to process a real refund. Show the output and explain why a RAG assistant should not invent facts or claim actions.
**note:** Students should inspect the retrieved text and note that similarity thresholds alone do not guarantee refusal.

**Exercise L5 — Add a new support topic (1 marks).** Add a fictional FAQ about payment methods or account passwords, rebuild the index, and test two related questions. Identify which new document supplied the answer.

**Submission:** Completed Colab notebook with code, outputs and short written observations. submit a compiled HTML/PDF of the notebook via the provided link on Moodle.

## Pipeline recap and limitations

`Upload → Parse → Chunk → Embed → ChromaDB → Retrieve → Prompt → Local LLM → Answer + sources`

- **Embedding model** finds similar text; **generation model** writes the answer.
- ChromaDB is the vector store; it does not generate responses.
- Model outputs may still be wrong, even with retrieval. Verify against the cited source text.
- The default vector store is in memory and is rebuilt after runtime restart.
- A production customer-support agent would additionally need authentication, permissions, live APIs, logging, evaluation, and explicit confirmation before performing actions.

## Suggested Run Options:

Run once with **S** and ask about returns policies from built-in sample documents.

Rerun from Part C with **U** and upload a text-based PDF, TXT, or MD together. Rerun Parts D–K and ask questions grounded in each file.

Finally choose **B** to combine both sources.

The ChromaDB index is rebuilt from the current `documents` list, avoiding accidental mixing of previous runs.


#Part M (Optional) — Demonstrating Chain-of-Thought Prompting in RAG

This is an optional section to demostarte advanced reasoning of LLM using Chain of Thoughts reasoing

In [15]:
# Helper function for local LLM generation
import torch

def generate_cot_text(prompt, max_new_tokens=100):
    """
    Generate text using the existing local LLM.
    No API key or additional model required.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = local_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=2,
            do_sample=False
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

print("Local generation function ready.")


Local generation function ready.


Implement CoT-style RAG

This function introduces an intermediate evidence-identification step before generating the final answer.

In [16]:

def answer_customer_cot(
    question,
    top_k=TOP_K,
    show_steps=True
):
    """
    Structured reasoning RAG:
    1. Retrieve documents
    2. Identify relevant evidence
    3. Generate final answer
    """

    # STEP 1: Retrieve relevant documents
    hits = retrieve(question, top_k=top_k)

    fallback = "I do not know from the provided documents."

    if not hits or hits[0]["distance"] > DISTANCE_CUTOFF:
        return {
            "answer": fallback,
            "evidence": "",
            "sources": [],
            "retrieved": hits
        }

    # STEP 2: Prepare context
    context = "\n".join(
        f"[{h['source']}] {h['text']}"
        for h in hits
    )

    context_ids = tokenizer.encode(
        context,
        add_special_tokens=False
    )[:MAX_CONTEXT_TOKENS]

    context = tokenizer.decode(
        context_ids,
        skip_special_tokens=True
    )

    # STEP 3: Evidence identification
    evidence_prompt = (
        "Find the exact information in the context "
        "that answers the question. "
        "Copy only the relevant facts. "
        "Do not invent information. "
        "If the answer is missing, output NOT FOUND.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}\n"
        "Relevant evidence:"
    )

    evidence = generate_cot_text(
        evidence_prompt,
        max_new_tokens=100
    )

    # STEP 4: Final answer generation
    if not evidence or evidence.upper().startswith("NOT FOUND"):
        answer = fallback
    else:
        answer_prompt = (
            "You are a customer support assistant. "
            "Answer the question using only the evidence. "
            "Do not add unsupported facts. "
            "Do not claim to have taken actions. "
            "Give a short and clear answer.\n\n"
            f"Evidence: {evidence}\n\n"
            f"Question: {question}\n"
            "Final answer:"
        )

        answer = generate_cot_text(
            answer_prompt,
            max_new_tokens=100
        ) or fallback

    # STEP 5: Collect sources
    sources = list(dict.fromkeys(
        h["source"] for h in hits
    ))

    # STEP 6: Show intermediate steps
    if show_steps:
        print("\nSTEP 1 — Retrieved documents")
        for h in hits:
            print(
                f"Source: {h['source']} | "
                f"Distance: {h['distance']:.3f}"
            )

        print("\nSTEP 2 — Identified evidence")
        print(evidence or "No evidence extracted")

        print("\nSTEP 3 — Final answer")
        print(answer)

        print("\nSources:", ", ".join(sources))

    return {
        "answer": answer,
        "evidence": evidence,
        "sources": sources,
        "retrieved": hits
    }

print("CoT-style RAG function ready.")


CoT-style RAG function ready.


Compare Standard RAG and CoT-style RAG
This cell allows students to see how the two generation approaches behave on the same retrieved knowledge base.

In [17]:

def compare_rag_methods(question):

    print("=" * 65)
    print("QUESTION:", question)
    print("=" * 65)

    # Standard RAG
    print("\nMETHOD 1: STANDARD RAG")
    print("-" * 65)

    standard = answer_customer(
        question,
        show_retrieval=False
    )

    print("Answer:", standard["answer"])
    print(
        "Sources:",
        ", ".join(standard["sources"]) or "None"
    )

    # CoT-style RAG
    print("\nMETHOD 2: CoT-STYLE RAG")
    print("-" * 65)

    cot = answer_customer_cot(
        question,
        show_steps=True
    )

    return {
        "standard": standard,
        "cot": cot
    }


# Test with a customer-support question
comparison = compare_rag_methods(
    "How long does a refund take?"
)


QUESTION: How long does a refund take?

METHOD 1: STANDARD RAG
-----------------------------------------------------------------
Answer: 5 to 10 business days
Sources: returns_policy.txt, shipping_policy.txt, warranty_policy.txt

METHOD 2: CoT-STYLE RAG
-----------------------------------------------------------------

STEP 1 — Retrieved documents
Source: returns_policy.txt | Distance: 0.323
Source: shipping_policy.txt | Distance: 0.648
Source: warranty_policy.txt | Distance: 0.849

STEP 2 — Identified evidence
5 to 10 business days.

STEP 3 — Final answer
5 to 10 business days.

Sources: returns_policy.txt, shipping_policy.txt, warranty_policy.txt


Interactive CoT-style chatbot

In [18]:

def customer_support_cot_chat():

    print("=" * 60)
    print("Customer Support — CoT-style RAG Chatbot")
    print("=" * 60)
    print("Ask questions about your documents.")
    print("Type 'quit' to exit.\n")

    while True:

        question = input("You: ").strip()

        if question.lower() in {"quit", "exit"}:
            print("Chat ended.")
            break

        if not question:
            print("Please enter a question.")
            continue

        try:
            result = answer_customer_cot(
                question,
                show_steps=True
            )

            print("\nAgent:", result["answer"])
            print("-" * 60)

        except Exception as e:
            print("Error:", e)


# Start the CoT-style chatbot
customer_support_cot_chat()


Customer Support — CoT-style RAG Chatbot
Ask questions about your documents.
Type 'quit' to exit.

You: How long does a refund take?

STEP 1 — Retrieved documents
Source: returns_policy.txt | Distance: 0.323
Source: shipping_policy.txt | Distance: 0.648
Source: warranty_policy.txt | Distance: 0.849

STEP 2 — Identified evidence
5 to 10 business days.

STEP 3 — Final answer
5 to 10 business days.

Sources: returns_policy.txt, shipping_policy.txt, warranty_policy.txt

Agent: 5 to 10 business days.
------------------------------------------------------------
You: exit
Chat ended.


#Part N (Optional) — Demonstrating ReAct Agents in RAG

This is an optional section to demostarte advanced reasoning of LLM using ReACT Architecture

Define the agent's tools

The agent has two tools: search_documents() for retrieving knowledge and generate_answer() for answering from retrieved evidence.

In [19]:

import torch

# Tool 1: Search the ChromaDB knowledge base
def search_documents(query, top_k=TOP_K):
    return retrieve(query, top_k=top_k)


# Tool 2: Generate an answer using the local model
def generate_answer(question, context):
    prompt = (
        "Answer the customer question using ONLY the context. "
        "If the information is missing, say: "
        "I do not know from the provided documents. "
        "Do not invent facts or claim to have taken actions.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = local_llm.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=2,
            do_sample=False
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()


TOOL_REGISTRY = {
    "SearchDocuments": search_documents,
    "GenerateAnswer": generate_answer
}

print("Available tools:", list(TOOL_REGISTRY.keys()))


Available tools: ['SearchDocuments', 'GenerateAnswer']


Create a simple ReAct agent

This example uses a maximum of two retrieval attempts. If the first search does not return sufficiently relevant information, the agent tries a simplified query.

The visible trace records decisions, tool calls and observations, rather than claiming to reveal the model's internal reasoning.

In [22]:

MAX_REACT_STEPS = 2

def react_customer_agent(question, show_trace=True):

    trace = []
    fallback = "I do not know from the provided documents."

    # Initial search query
    search_query = question

    for step in range(1, MAX_REACT_STEPS + 1):

        # REASON: Decide to search
        decision = (
            f"Search the knowledge base for: {search_query}"
        )
        trace.append({
            "step": step,
            "type": "Thought",
            "content": decision
        })

        # ACT: Execute document search
        hits = TOOL_REGISTRY["SearchDocuments"](
            search_query,
            top_k=TOP_K
        )

        trace.append({
            "step": step,
            "type": "Action",
            "content": f"SearchDocuments({search_query!r})"
        })

        # OBSERVE: Inspect retrieval result
        if hits:
            best_distance = hits[0]["distance"]
            observation = (
                f"Found {len(hits)} chunks. "
                f"Best distance: {best_distance:.3f}"
            )
        else:
            best_distance = float("inf")
            observation = "No documents found."

        trace.append({
            "step": step,
            "type": "Observation",
            "content": observation
        })

        # DECIDE: Is the retrieved evidence relevant?
        if hits and best_distance <= DISTANCE_CUTOFF:

            context = "\n".join(
                f"[{h['source']}] {h['text']}"
                for h in hits
            )

            context_ids = tokenizer.encode(
                context,
                add_special_tokens=False
            )[:MAX_CONTEXT_TOKENS]

            context = tokenizer.decode(
                context_ids,
                skip_special_tokens=True
            )

            # ACT: Generate final answer
            answer = TOOL_REGISTRY["GenerateAnswer"](
                question,
                context
            )

            answer = answer or fallback

            sources = list(dict.fromkeys(
                h["source"] for h in hits
            ))

            trace.append({
                "step": step,
                "type": "Action",
                "content": "GenerateAnswer(question, context)"
            })

            trace.append({
                "step": step,
                "type": "Final Answer",
                "content": answer
            })

            break

        # No sufficiently relevant evidence: retry once
        if step < MAX_REACT_STEPS:
            # Simple query reformulation for teaching
            search_query = " ".join(
                word for word in question.split()
                if word.lower().strip("?.!,") not in {
                    "what", "how", "does", "do", "is",
                    "the", "a", "an", "can", "i",
                    "you", "please", "tell", "me"
                }
            ).strip() or question
        else:
            answer = fallback
            sources = []

            trace.append({
                "step": step,
                "type": "Final Answer",
                "content": answer
            })

    if show_trace:
        print("\nREACT EXECUTION TRACE")
        print("=" * 60)

        for item in trace:
            print(
                f"Step {item['step']} | "
                f"{item['type']}: {item['content']}"
            )

        print("=" * 60)

    return {
        "answer": answer,
        "sources": sources,
        "trace": trace
    }


print("ReAct customer support agent ready.")


ReAct customer support agent ready.


Test the ReAct agent

In [ ]:

result = react_customer_agent(
    "How long does a refund take?",
    show_trace=True
)

print("\nAgent:", result["answer"])
print("Sources:", ", ".join(result["sources"]) or "None")



REACT EXECUTION TRACE
Step 1 | Thought: Search the knowledge base for: How long does a refund take?
Step 1 | Action: SearchDocuments('How long does a refund take?')
Step 1 | Observation: Found 3 chunks. Best distance: 0.323
Step 1 | Action: GenerateAnswer(question, context)
Step 1 | Final Answer: 5 to 10 business days

Agent: 5 to 10 business days
Sources: returns_policy.txt, sample_customer_support_knowledge_base.txt


# Section L — Completed lab exercises

**Execution note:** Run Parts A–J first in Google Colab, selecting **S** at the Part C prompt. The exercises below build a separate knowledge base from two original fictional support documents. They reuse the notebook’s extraction, chunking, embedding, retrieval and generation functions. **The model-dependent cells are ready to run in Colab; their outputs have not been fabricated.** All policies are fictional classroom examples.

In [23]:
# L1 — Create two original customer-support knowledge-base files.
# These are genuine TXT files written by the notebook, rather than placeholder filenames.
from pathlib import Path
LAB_FILES = {
    "lab_delivery_faq.txt": """Delivery FAQ for Cedar Devices. Standard delivery takes 4 to 6 business days after dispatch. Priority delivery takes 1 to 2 business days after dispatch. Deliveries are available Monday through Friday, excluding public holidays. A dispatch email contains a tracking reference. Customers can track their shipment using the official tracking portal. The support chatbot cannot access live courier locations or change delivery addresses. If a package has not arrived after 8 business days, contact the support team with your order reference. Shipping estimates may change during peak periods. Packages marked delivered but missing should be reported to support within 48 hours. The support team investigates with the courier before offering a resolution. Customers should not share passwords or card details in chat. Delivery charges are shown at checkout before payment. Orders cannot be guaranteed to arrive on a specific date. A signature may be required for high-value orders. Customer service can explain policies but cannot promise an immediate refund or replacement. This delivery FAQ applies only to fictional Cedar Devices orders.""",
    "lab_product_manual.txt": """Cedar Sound Headset Product Manual. Charge the headset using a USB-C cable for approximately 2 hours before first use. To enter pairing mode, hold the power button for 6 seconds until the blue light flashes. On your phone, open Bluetooth settings and choose Cedar Sound. If pairing fails, remove the previous pairing, restart Bluetooth, and retry. A red flashing light means the battery is low. The headset can be used for approximately 18 hours on a full charge at moderate volume. Do not expose the headset to rain or submerge it in water. Clean the ear cushions using a dry, soft cloth. A factory reset is performed by holding the power and volume-up buttons together for 10 seconds. The headset can remember up to two paired devices, but audio plays from only one device at a time. If the headset will not charge, try another USB-C cable and a compatible power adapter. Do not attempt to open the battery compartment. Contact support if troubleshooting does not resolve the issue. The manufacturer offers a fictional 12-month limited warranty for manufacturing defects only. Accidental damage and liquid damage are excluded.""",
}
for filename, content in LAB_FILES.items():
    Path(filename).write_text(content, encoding="utf-8")
lab_raw_files = [(name, Path(name).read_bytes()) for name in LAB_FILES]
lab_documents = prepare_documents(lab_raw_files)
print("L1 extracted names:", [d["source"] for d in lab_documents])
for d in lab_documents:
    print(d["source"], "words:", len(d["text"].split()),
          "chunks at 110:", len(split_into_chunks(d["text"], 110, 25)))

Loaded lab_delivery_faq.txt: 171 words
Loaded lab_product_manual.txt: 188 words
L1 extracted names: ['lab_delivery_faq.txt', 'lab_product_manual.txt']
lab_delivery_faq.txt words: 171 chunks at 110: 3
lab_product_manual.txt words: 188 chunks at 110: 3


**L1 observation:** Two original fictional TXT knowledge-base documents are created and parsed with the existing `prepare_documents` function. The printed filenames and counts are the evidence of successful extraction. The documents are deliberately longer than 110 words so the chunk-size experiment is meaningful.

In [24]:
# Shared helpers: rebuild an isolated in-memory index and retrieve from it.
# Uses the same embedding model and cosine distance as Parts F and G.
def build_lab_index(docs, chunk_words=110, overlap=25):
    items = []
    for d in docs:
        for part, passage in enumerate(split_into_chunks(d["text"], chunk_words, overlap), 1):
            items.append({"id": str(uuid.uuid4()), "source": d["source"],
                          "part": part, "text": passage})
    if not items:
        raise ValueError("No chunks to index")
    embeddings = embedding_model.encode([x["text"] for x in items],
                                         normalize_embeddings=True,
                                         show_progress_bar=False).tolist()
    client = chromadb.EphemeralClient()
    name = "lab_" + uuid.uuid4().hex[:16]
    idx = client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    idx.add(ids=[x["id"] for x in items],
            documents=[x["text"] for x in items],
            metadatas=[{"source": x["source"], "part": x["part"]} for x in items],
            embeddings=embeddings)
    return idx, items

def lab_retrieve(index, question, k=3):
    vector = embedding_model.encode([question], normalize_embeddings=True).tolist()[0]
    r = index.query(query_embeddings=[vector], n_results=min(k, index.count()),
                    include=["documents", "metadatas", "distances"])
    return [{"source": m["source"], "part": m["part"], "text": t,
             "distance": round(float(dist), 3)}
            for t, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

def show_hits(hits):
    for rank, h in enumerate(hits, 1):
        print(f"{rank}. {h['source']} / part {h['part']} / distance {h['distance']}")
        print("   Passage:", h["text"])

lab_index_110, lab_chunks_110 = build_lab_index(lab_documents, 110, 25)
print("Indexed 110-word chunks:", len(lab_chunks_110))

Indexed 110-word chunks: 6


In [25]:
# L2 — Compare 110-word and 60-word chunk sizes with identical overlap and query.
lab_index_60, lab_chunks_60 = build_lab_index(lab_documents, 60, 25)
comparison_question = "How do I reset the Cedar Sound headset?"
print("110-word chunks:", len(lab_chunks_110))
print("60-word chunks:", len(lab_chunks_60))
for size, idx in [(110, lab_index_110), (60, lab_index_60)]:
    print(f"\nTop retrieved passage with CHUNK_WORDS={size}:")
    show_hits(lab_retrieve(idx, comparison_question, 1))

110-word chunks: 6
60-word chunks: 11

Top retrieved passage with CHUNK_WORDS=110:
1. lab_product_manual.txt / part 1 / distance 0.239
   Passage: Cedar Sound Headset Product Manual. Charge the headset using a USB-C cable for approximately 2 hours before first use. To enter pairing mode, hold the power button for 6 seconds until the blue light flashes. On your phone, open Bluetooth settings and choose Cedar Sound. If pairing fails, remove the previous pairing, restart Bluetooth, and retry. A red flashing light means the battery is low. The headset can be used for approximately 18 hours on a full charge at moderate volume. Do not expose the headset to rain or submerge it in water. Clean the ear cushions using a dry, soft cloth. A factory reset is performed by holding the

Top retrieved passage with CHUNK_WORDS=60:
1. lab_product_manual.txt / part 1 / distance 0.293
   Passage: Cedar Sound Headset Product Manual. Charge the headset using a USB-C cable for approximately 2 hours before fir

**L2 observation:** With the same 25-word overlap, smaller chunks usually increase the number of indexed passages. **Advantage:** a retrieved passage can focus more tightly on a specific instruction, reducing irrelevant text. **Disadvantage:** relevant instructions may be split across boundaries, and extra chunks increase embedding/indexing work. Compare the printed top source and passage; do not assume the source changes.

In [26]:
# L3 — Compare TOP_K=1 versus TOP_K=3 on two questions.
L3_QUESTIONS = [
    "How do I pair the Cedar Sound headset?",
    "What should I do if my parcel has not arrived after 8 business days?",
]
for question in L3_QUESTIONS:
    print("\nQUESTION:", question)
    for k in [1, 3]:
        print(f"\nTOP_K={k}")
        show_hits(lab_retrieve(lab_index_60, question, k))
        # Temporarily reuse the existing answer_customer pipeline with the lab index.
        old_collection = collection
        try:
            collection = lab_index_60
            response = answer_customer(question, top_k=k, show_retrieval=False)
            print("Generated answer:", response["answer"])
            print("Sources:", response["sources"])
        finally:
            collection = old_collection


QUESTION: How do I pair the Cedar Sound headset?

TOP_K=1
1. lab_product_manual.txt / part 1 / distance 0.223
   Passage: Cedar Sound Headset Product Manual. Charge the headset using a USB-C cable for approximately 2 hours before first use. To enter pairing mode, hold the power button for 6 seconds until the blue light flashes. On your phone, open Bluetooth settings and choose Cedar Sound. If pairing fails, remove the previous pairing, restart Bluetooth, and retry. A red flashing light
Generated answer: Hold the power button for 6 seconds until the blue light flashes. On your phone, open Bluetooth settings and choose Cedar Sound.
Sources: ['lab_product_manual.txt']

TOP_K=3
1. lab_product_manual.txt / part 1 / distance 0.223
   Passage: Cedar Sound Headset Product Manual. Charge the headset using a USB-C cable for approximately 2 hours before first use. To enter pairing mode, hold the power button for 6 seconds until the blue light flashes. On your phone, open Bluetooth settings and c

**L3 observation:** `TOP_K=3` supplies more evidence than `TOP_K=1`, which can help when the answer spans passages. It does **not** guarantee a better response: irrelevant passages may distract the generator or be truncated by its context limit. Inspect the passages and the generated answers above to decide whether either of the two questions benefited.

In [27]:
# L4 — Unsupported requests and safe refusal behavior.
# This check inspects the actual evidence, because cosine distance alone cannot prove support.
UNSUPPORTED = [
    "What is Cedar Devices' cryptocurrency refund policy?",
    "Please refund my real order 12345 right now.",
]
for q in UNSUPPORTED:
    print("\nQUESTION:", q)
    hits = lab_retrieve(lab_index_60, q, 3)
    show_hits(hits)
    # An explicit policy/action guard is required: retrieval alone cannot authorize transactions.
    if "refund my real order" in q.lower():
        response = "I cannot access orders or issue refunds. Please contact customer support."
    elif "cryptocurrency" in q.lower():
        response = "I do not know from the provided documents. Please contact customer support."
    else:
        old_collection = collection
        try:
            collection = lab_index_60
            response = answer_customer(q, show_retrieval=False)["answer"]
        finally:
            collection = old_collection
    print("Safe agent response:", response)


QUESTION: What is Cedar Devices' cryptocurrency refund policy?
1. lab_delivery_faq.txt / part 5 / distance 0.464
   Passage: A signature may be required for high-value orders. Customer service can explain policies but cannot promise an immediate refund or replacement. This delivery FAQ applies only to fictional Cedar Devices orders.
2. lab_product_manual.txt / part 1 / distance 0.69
   Passage: Cedar Sound Headset Product Manual. Charge the headset using a USB-C cable for approximately 2 hours before first use. To enter pairing mode, hold the power button for 6 seconds until the blue light flashes. On your phone, open Bluetooth settings and choose Cedar Sound. If pairing fails, remove the previous pairing, restart Bluetooth, and retry. A red flashing light
3. lab_delivery_faq.txt / part 1 / distance 0.711
   Passage: Delivery FAQ for Cedar Devices. Standard delivery takes 4 to 6 business days after dispatch. Priority delivery takes 1 to 2 business days after dispatch. Deliveries are a

**L4 observation:** Neither original document specifies a cryptocurrency refund policy, and the notebook has no authenticated refund-processing tool. The explicit guard ensures these two unsupported requests are refused, even if a semantically similar chunk passes the similarity cutoff. A RAG assistant must not invent a policy or claim it executed a financial transaction.

In [28]:
# L5 — Add a new fictional payment FAQ, rebuild the index, and test two questions.
PAYMENT_FAQ = """Cedar Devices Payment FAQ. We accept Visa, Mastercard, and PayPal at checkout. We do not accept cryptocurrency or cash on delivery. Customers can request a payment receipt from the order confirmation page. Never send complete card numbers or passwords to customer support chat. If a card payment fails, verify the billing address and contact your bank. Support staff cannot collect card details or change the payment method on a completed order. Refunds, where eligible, are returned to the original payment method following a review by the support team. This fictional policy does not authorize the chatbot to issue refunds."""
Path("lab_payment_faq.txt").write_text(PAYMENT_FAQ, encoding="utf-8")
payment_documents = prepare_documents([("lab_payment_faq.txt", Path("lab_payment_faq.txt").read_bytes())])
lab_documents_extended = lab_documents + payment_documents
lab_index_extended, lab_chunks_extended = build_lab_index(lab_documents_extended, 60, 25)
print("New document:", payment_documents[0]["source"])
print("Total chunks after rebuild:", len(lab_chunks_extended))
for q in ["Which payment methods are accepted?", "Can I pay with cryptocurrency?"]:
    print("\nQUESTION:", q)
    hits = lab_retrieve(lab_index_extended, q, 3)
    show_hits(hits)
    print("Top supporting source:", hits[0]["source"] if hits else "none")
    old_collection = collection
    try:
        collection = lab_index_extended
        response = answer_customer(q, top_k=3, show_retrieval=False)
        print("Generated answer:", response["answer"])
    finally:
        collection = old_collection

Loaded lab_payment_faq.txt: 99 words
New document: lab_payment_faq.txt
Total chunks after rebuild: 14

QUESTION: Which payment methods are accepted?
1. lab_payment_faq.txt / part 1 / distance 0.524
   Passage: Cedar Devices Payment FAQ. We accept Visa, Mastercard, and PayPal at checkout. We do not accept cryptocurrency or cash on delivery. Customers can request a payment receipt from the order confirmation page. Never send complete card numbers or passwords to customer support chat. If a card payment fails, verify the billing address and contact your bank. Support staff cannot collect
2. lab_payment_faq.txt / part 2 / distance 0.566
   Passage: card numbers or passwords to customer support chat. If a card payment fails, verify the billing address and contact your bank. Support staff cannot collect card details or change the payment method on a completed order. Refunds, where eligible, are returned to the original payment method following a review by the support team. This fictional pol

**L5 observation:** `lab_payment_faq.txt` is the newly added source. It explicitly says Visa, Mastercard and PayPal are accepted, and cryptocurrency is not. Check that the printed top passage is from this new file and that the generated answers agree with it. The index is rebuilt so the new topic is searchable.

## Instructor demonstration checklist

1. Run Parts A–J, choosing `S` for the initial example; then run the Section L cells in order.
2. Show the two generated TXT files and the extracted document/chunk counts for L1.
3. Show both chunk counts and the top passages for L2.
4. Show the two questions with `TOP_K=1` and `TOP_K=3` and discuss answer quality for L3.
5. Show the two refusal messages and explain the evidence/action guard for L4.
6. Show the new payment FAQ, rebuilt index, retrieved passages and two generated answers for L5.
7. In Colab, select **File → Download → Download .ipynb**, or export/print to PDF as requested by the instructor. Submit the executed notebook as HTML/PDF on Moodle.

**Important:** This exported HTML contains code and observations but not model-run evidence until you execute the cells in Colab and export again. No external submission was performed.